In [1]:
print("Hello")

Hello


In [2]:
!pip install openai python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://packagefeedproxy.microsoft.io/pypi/simple/


In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

print("Key loaded:", bool(api_key))

Key loaded: True


In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load variables from .env
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    raise ValueError("OPENROUTER_API_KEY not found in .env")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

response = client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {
            "role": "user",
            "content": "Explain BGP in simple terms for a network engineer."
        }
    ]
)

print(response.choices[0].message.content)

User Safety: safe


In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

network_log = """
RP/0/RP0/CPU0:R1#show bgp summary

Neighbor        AS      State
10.1.1.2        65002   Established
10.1.1.6        65003   Active
10.1.1.10       65004   Idle
"""

prompt = f"""
You are a senior network engineer.

Analyze the following BGP output:

{network_log}

Tell me:

1. What looks normal?
2. What looks abnormal?
3. Possible root causes.
4. Commands I should run next.
5. Recommended troubleshooting sequence.
"""

response = client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)

# BGP Summary Analysis — R1

## 1. What Looks Normal ✅

- **10.1.1.2 (AS 65002) — Established**: This peer is fully operational. Routes are being exchanged, keepalives are flowing, and the TCP session (port 179) is healthy. This also confirms the local interface, routing to the peer, and basic BGP process are functional.

## 2. What Looks Abnormal ⚠️

- **10.1.1.6 (AS 65003) — Active**: R1 is actively trying to **initiate** a TCP connection to the peer but failing repeatedly. This is a transitional state — BGP will keep retrying with exponential back-off. This strongly suggests the remote end is **not accepting connections** or TCP 179 is being blocked.
- **10.1.1.10 (AS 65004) — Idle**: BGP is **not even attempting** to connect. This points to a configuration or administrative issue rather than a transient connectivity problem.

> Having one Established and two non-established peers on the same subnet suggests the issue is **peer-specific**, not a universal local problem.

## 3. Possi

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load .env
load_dotenv()

# Get API key
api_key = os.getenv("OPENROUTER_API_KEY")

# Check key
if not api_key:
    raise ValueError("OPENROUTER_API_KEY not found in .env")

print("✅ API key loaded")

# Create OpenRouter client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

# Send request
response = client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {
            "role": "user",
            "content": "Explain BGP in simple terms for a network engineer."
        }
    ]
)

# Print answer
print("\nAI Response:\n")
print(response.choices[0].message.content)

✅ API key loaded

AI Response:

Here's BGP explained in simple, practical terms **specifically for a network engineer** (assuming you know IP, routing basics, AS numbers, and IGPs like OSPF/EIGRP):

---

### 🌐 **BGP in a Nutshell**
**BGP (Border Gateway Protocol) is the "diplomacy protocol" of the internet.**  
It doesn’t find the *shortest* path like OSPF/IS-IS. Instead, it lets **Autonomous Systems (ASes)** (think: ISPs, large enterprises, cloud providers) **exchange routing information and *negotiate* which paths to use based on business relationships, policies, and technical constraints.**

---

### 🔑 **Key Concepts (Engineer-Focused)**
1. **It’s Path-Vector, Not Link-State**  
   - Unlike OSPF (which shares *link states* to build a full topology map), BGP shares *entire AS paths* (e.g., `AS_PATH: 65001 65002 65003`).  
   - **Why?** To prevent loops *between* ASes (IGPs handle loops *inside* an AS). You only need to know the sequence of ASes, not every router inside them.

2. **It

In [5]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

MODEL = "openrouter/free"

In [6]:
interface_output = """
Ethernet1/1 is up, line protocol is up
  MTU 9216 bytes
  5 minute input rate 850000000 bits/sec
  5 minute output rate 920000000 bits/sec
  125 input errors
  87 CRC errors
  0 frame errors
  15432 output drops
"""

prompt = f"""
You are a senior network engineer.

Analyze this interface output:

{interface_output}

Explain:
1. What looks normal
2. What looks abnormal
3. Possible causes
4. What I should check next
5. Give troubleshooting commands
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

## Analysis of Ethernet1/1

### 1. What looks normal
- **up/up** — Physical and data-link layers are operational.
- **MTU 9216** — Jumbo frames enabled (typical for data center/storage).
- **0 frame errors** — Good; rules out framing/mac-level issues.
- Symmetric, high throughput (850 Mbit/s in / 920 Mbit/s out) suggests the link is carrying heavy but stable traffic.

### 2. What looks abnormal
- **87 CRC errors + 125 total input errors** — Physical-layer corruption. CRCs are not caused by congestion; they indicate bad cabling, optics, or NIC/hardware issues.
- **15,432 output drops** — The egress side is discarding packets. This points to congestion, buffer exhaustion, or QoS drops.
- Utilization looks high (920 Mbit/s out). If this is a **1 Gbps** link, it’s saturated; if **10 Gbps**, you still have microburst/buffer issues.

### 3. Possible causes
| Symptom | Likely cause |
|---|---|
| CRC errors | Bad/damaged cable, faulty SFP/optic, duplex mismatch, dirty connector, bad NIC port |

In [7]:
bgp_output = """
RP/0/RP0/CPU0:R1# show bgp summary

Neighbor        AS      MsgRcvd    MsgSent    State
10.1.1.2        65002   123455     125432     Established
10.1.1.6        65003   0          0          Active
10.1.1.10       65004   0          0          Idle
"""

prompt = f"""
Act as a senior Cisco IOS XR network engineer.

Analyze this BGP output:

{bgp_output}

For every neighbor explain:

- Current state
- Whether it is healthy
- Possible problem
- Commands to run next
- Most likely root causes

Give me a troubleshooting sequence.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

We in this scenario we have three neighbors:

 1. 10.1.1.2: established, healthy, no problem
 2. 10<unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><

In [8]:
problem = """
Client: 10.1.1.10
Server: 10.20.20.50
Server Port: TCP 443

Packet capture on client:

10.1.1.10 -> 10.20.20.50 SYN
10.1.1.10 -> 10.20.20.50 SYN retransmission
10.1.1.10 -> 10.20.20.50 SYN retransmission

No SYN-ACK is visible in the client capture.
"""

prompt = f"""
You are a senior network troubleshooting engineer.

Analyze this problem:

{problem}

Explain possible causes including:

- Client
- Network
- Firewall
- NAT
- Load balancer
- Server
- Asymmetric routing
- Packet loss

Then provide a troubleshooting workflow from client to server.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

**Problem Summary**  
- **Client**: `10.1.1.10`  
- **Server**: `10.20.20.50` (TCP 443)  
- **Observation on the client**: three outbound **SYN** packets followed by three retransmissions – **no SYN‑ACK ever appears in the capture**.  

The absence of a SYN‑ACK can be caused by anything that prevents the TCP three‑way handshake from completing: the server never replies, the reply never reaches the client, or the client capture simply never sees the inbound traffic. Below is a **senior‑engineer view** of the most likely culprits, grouped by functional area, followed by a **practical, linear troubleshooting workflow** that will let you confirm (or eliminate) each hypothesis.

---

## 1. Possible Causes  

| Area | Why it could stop the SYN‑ACK from reaching the client | Typical evidence / clues |
|------|--------------------------------------------------------|--------------------------|
| **Client** | • **TCP stack parameters** – e.g., `tcp_retries2`, `tcp_syncookies`, `tcp_mtu_discover

In [9]:
config = """
router bgp 65001
 neighbor 10.1.1.2
  remote-as 65002
  update-source Loopback0

interface Loopback0
 ipv4 address 192.168.1.1 255.255.255.255
"""

prompt = f"""
Review this Cisco IOS XR-style configuration:

{config}

Look for:

1. BGP problems
2. Missing configuration
3. Reachability requirements
4. Update-source issues
5. Security concerns
6. Recommended verification commands

Do not assume the configuration is correct.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)



Here's a comprehensive review of the provided Cisco IOS XR-style BGP configuration:

## 1. BGP Problems

**Critical Issues:**
- **Missing BGP address-family configuration** - No address-family is specified, so no routes will be exchanged
- **No network statements or redistribution** - No routes are being advertised
- **Missing router-id** - Not explicitly configured, may cause instability
- **No route-target or VRF configuration** - Assuming this is for global BGP, but unclear if VRFs are needed

## 2. Missing Configuration

**Required additions:**
```ios
router bgp 65001
 address-family ipv4 unicast
  neighbor 10.1.1.2
   activate
   route-policy OUTBOUND in
   route-policy INBOUND out
 redistribute connected
 redistribute static
 router-id 192.168.1.1
 maximum-paths 4
 bgp log-neighbor-changes
```

**Security configuration:**
```ios
router bgp 65001
 neighbor 10.1.1.2
  password 7 <encrypted-password>
  ttl-security
```

## 3. Reachability Requirements

**Interface configuration ne

In [10]:
question = """
I want to check:

- BGP summary
- BGP neighbor
- interface counters
- interface errors
- LACP status
- LLDP neighbors
- routing table
- ARP table
"""

prompt = f"""
You are a multi-vendor network engineer.

For the following requirements:

{question}

Give equivalent commands for:

Cisco IOS XR
Juniper Junos
Arista EOS

Create a comparison table.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

Here are the equivalent commands for checking the requested items on Cisco IOS XR, Juniper Junos, and Arista EOS.

### Equivalent Commands

| Check Item | Cisco IOS XR | Juniper Junos | Arista EOS |
| :--- | :--- | :--- | :--- |
| **BGP Summary** | `show bgp summary` | `show bgp summary` | `show bgp summary` |
| **BGP Neighbor** | `show bgp neighbor` | `show bgp neighbor` | `show bgp neighbor` |
| **Interface Counters** | `show interfaces <interface-name> counters` | `show interface <int> counters` | `show interfaces <int> counters` |
| **Interface Errors** | `show interface <interface-name> errors` | `show interface <int> errors` | `show interfaces <int> errors` |
| **LACP Status** | `show lags` | `show lag` | `show lag` |
| **LLDP Neighbors** | `show lldp neighbors` | `show lldp neighbors` | `show lldp neighbors` |
| **Routing Table** | `show ip route` | `show ip route` | `show ip route` |
| **ARP Table** | `show arp` | `show arp` | `show arp` |

---

### Detailed Command Breakdown



In [11]:
prompt = """
Create a troubleshooting decision tree for:

BGP neighbor stuck in Active state.

Environment:
Cisco IOS XR

Start with Layer 1 and work upward.

Include:

Interface
ARP/ND
Routing
Ping
TCP 179
ACL/firewall
BGP configuration
ASN
Update source
Authentication
Timers

Show the troubleshooting sequence using ASCII arrows.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)



Here's a troubleshooting decision tree for BGP neighbor stuck in Active state on Cisco IOS XR:

```
START
│
├─ LAYER 1 (Physical)
│  ├─ Interface status: shutdown? → UN-SHUTDOWN
│  ├─ Interface line protocol down? → CHECK CABLE/SFP
│  └─ Interface errors? → CHECK HARDWARE/INTERFACE
│
├─ LAYER 2 (Data Link)
│  ├─ ARP/ND resolution?
│  │  ├─ IPv4: "show arp" → Static ARP? → ADD DYNAMIC ARP
│  │  └─ IPv6: "show ipv6 neighbors" → ND unreachable? → CHECK L2 SWITCH
│  └─ VLAN mismatch? → ALIGN VLAN CONFIG
│
├─ LAYER 3 (Network)
│  ├─ IP connectivity?
│  │  ├─ Ping neighbor IP → FAIL? → CHECK ROUTING
│  │  └─ Ping source IP → FAIL? → CHECK INTERFACE IP
│  └─ Routing table?
│     ├─ Route to neighbor exists? → ADD STATIC/IGP ROUTE
│     └─ Route via correct next-hop? → CHECK ROUTING PROTOCOL
│
├─ LAYER 4 (Transport)
│  ├─ TCP 179 reachable?
│  │  ├─ "show tcp brief" → No TCP listening? → CHECK BGP CONFIG
│  │  └─ TCP connect fails? → CHECK ACL/FIREWALL
│  └─ ACL/firewall?
│     ├─ Port 179 b

In [12]:
incident = """
10:01:02 Interface Ethernet1/1 down
10:01:03 BFD neighbor down
10:01:04 ISIS adjacency down
10:01:05 BGP routes withdrawn
10:01:07 Traffic loss detected
10:01:12 Interface Ethernet1/1 up
10:01:13 BFD neighbor up
10:01:15 ISIS adjacency restored
10:01:20 Traffic recovered
"""

prompt = f"""
Analyze this network incident:

{incident}

Create:

1. Incident timeline
2. First failure
3. Cascading failures
4. Likely root cause
5. Customer impact
6. Recovery sequence
7. What monitoring should be added
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

**1. Incident Timeline**

| Time (hh:mm:ss) | Event |
|-----------------|-------|
| **10:01:02** | Interface **Ethernet1/1** goes **down** (link loss / admin shutdown). |
| **10:01:03** | **BFD** neighbor on that interface detects the link loss → **BFD neighbor down**. |
| **10:01:04** | **IS‑IS** adjacency on the same link collapses (no hello/receive). |
| **10:01:05** | **BGP** routes are withdrawn because the underlying IGP (IS‑IS) is down and the BGP session can’t keep the session alive. |
| **10:01:07** | **Traffic loss detected** by monitoring tools (e.g., flow counters, ping failures). |
| **10:01:12** | **Ethernet1/1** comes back **up** (physical reconnection or auto‑recovery). |
| **10:01:13** | **BFD** neighbor sees the link up → **BFD neighbor up**. |
| **10:01:15** | **IS‑IS** adjacency is re‑established (Hello/DBD exchange succeeds). |
| **10:01:20** | **Traffic** is fully recovered (flows resume, counters return to normal). |

---

**2. First Failure**

- **Primary failur

In [13]:
import json

output = """
Ethernet1/1 is up
Input errors: 150
CRC errors: 120
Output drops: 25000
"""

prompt = f"""
Analyze:

{output}

Return ONLY valid JSON using this structure:

{{
    "interface_status": "",
    "severity": "",
    "problems": [],
    "possible_causes": [],
    "recommended_commands": []
}}
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

result = response.choices[0].message.content

print(result)

```json
{
    "interface_status": "Up",
    "severity": "High",
    "problems": [
        "CRC errors (120) indicating physical layer corruption",
        "Input errors (150) suggesting frame reception issues",
        "High output drops (25000) indicating severe congestion or interface saturation",
        "Likely duplex/speed mismatch causing CRC errors"
    ],
    "possible_causes": [
        "Damaged or improperly terminated cables",
        "Duplex mismatch (half-duplex on one end, full-duplex on other)",
        "Speed mismatch between connected devices",
        "Oversubscribed or congested link",
        "Faulty NIC on the connected device",
        "Broadcast storm or excessive traffic overwhelming the interface",
        "QoS misconfiguration causing egress queue drops"
    ],
    "recommended_commands": [
        "show interface Ethernet1/1",
        "show interface Ethernet1/1 counters errors",
        "show interface Ethernet1/1 status",
        "show interface Ethernet1/1

In [14]:
import subprocess

target = "8.8.8.8"

result = subprocess.run(
    ["ping", "-n", "4", target],
    capture_output=True,
    text=True
)

ping_output = result.stdout

print("PING OUTPUT")
print(ping_output)

prompt = f"""
You are a network engineer.

Analyze this Windows ping result:

{ping_output}

Determine:

- Reachability
- Packet loss
- Latency
- Any abnormal behavior
- Recommended next tests
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print("\nAI ANALYSIS")
print(response.choices[0].message.content)

PING OUTPUT

Pinging 8.8.8.8 with 32 bytes of data:
Request timed out.
Request timed out.
Request timed out.
Request timed out.

Ping statistics for 8.8.8.8:
    Packets: Sent = 4, Received = 0, Lost = 4 (100% loss),


AI ANALYSIS
## Quick Summary
| Metric | Result | Interpretation |
|--------|--------|----------------|
| **Reachability** | **Not reachable** (all 4 packets timed‑out) | The host **8.8.8.8** (Google DNS) is not responding to ICMP Echo Requests from this Windows machine. |
| **Packet loss** | **100 %** (0/4 received) | Every packet sent to the target was dropped before a reply could be generated. |
| **Latency** | **Not measurable** (no replies) | Ping‑time statistics (min/avg/max) are unavailable because the round‑trip time never completed. |
| **Abnormal behavior** | **Complete ICMP silence** | This is unusual for a standard Internet‑wide target like 8.8.8.8, which normally returns replies (or at least a mix of successes/timeouts). |
| **Likely causes** | • Firewall / A

In [15]:
import socket

website = "www.microsoft.com"

try:
    ip = socket.gethostbyname(website)
    result = f"DNS resolution successful. {website} resolved to {ip}"
except Exception as e:
    result = f"DNS resolution failed: {e}"

print(result)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": f"""
Analyze this DNS test:

{result}

Explain what it tells us and what network tests should be performed next.
"""
        }
    ]
)

print(response.choices[0].message.content)

DNS resolution successful. www.microsoft.com resolved to 6.6.0.84
This DNS test result (**"DNS resolution successful. www.microsoft.com resolved to 6.6.0.84"**) contains a **critical anomaly** that indicates a significant problem, despite the "successful" resolution message. Here's a breakdown of what it tells us and the essential next steps:

### 🔍 What This Result Tells Us
1. **DNS Resolution *Technically* Worked (But Returned Wrong Data):**
   - The DNS client successfully queried a DNS server and received a response (no timeout or SERVFAIL).
   - **However, the IP address `6.6.0.84` is categorically incorrect for `www.microsoft.com`.**
     - Microsoft's legitimate IPs for `www.microsoft.com` fall within ranges like:
       - `13.107.0.0/16` (e.g., `13.107.21.200`, `13.107.42.14`)
       - `40.0.0.0/8` (e.g., `40.112.72.205`, `40.126.32.0`)
       - `52.0.0.0/8` (Azure, less common for the main www site)
     - `6.6.0.84` belongs to the **`6.0.0.0/8` block**, which is **allocated t

In [16]:
import socket

host = "www.google.com"
port = 443

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.settimeout(5)

try:
    sock.connect((host, port))

    result = f"""
TCP connection successful.

Destination: {host}
Port: {port}
"""

except Exception as e:

    result = f"""
TCP connection failed.

Destination: {host}
Port: {port}

Error:
{e}
"""

finally:
    sock.close()

print(result)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": f"""
You are a network engineer.

Analyze this TCP connectivity test:

{result}

Explain what this proves about:

DNS
IP connectivity
TCP
Firewall
Destination service

Also explain what this test does NOT prove.
"""
        }
    ]
)

print(response.choices[0].message.content)


TCP connection successful.

Destination: www.google.com
Port: 443

## Analysis of Successful TCP Connection to www.google.com:443

### What This Proves

| Layer | Proven | Caveat |
|-------|--------|--------|
| **DNS** | Hostname resolved to an IP address at test time | Only proves this specific query succeeded; doesn't validate DNSSEC, TTL, or other domains |
| **IP Connectivity** | Bidirectional IP routing exists between source and destination | Proves reachability, not path quality, MTU, or ICMP permeability |
| **TCP** | 3-way handshake completed; destination port 443 is open | Only proves session establishment, not throughput, windowing, or congestion control |
| **Firewall** | Stateful/stateless rules permit this specific flow | Proves allow-policy for this dst IP:port, not other ports/protocols |
| **Destination Service** | A process is listening and accepting TCP connections on 443 | Does not confirm it's actually Google, or that HTTP/TLS is functional |

### What This Does NO

Absolutely. Now that the basic OpenRouter call works, you can turn the notebook into a small **AI networking lab**. I’d test these in roughly this order:

1. CLI output analysis
2. BGP troubleshooting
3. Interface/drop analysis
4. TCP troubleshooting
5. Log-file analysis
6. Configuration review
7. Cisco vs Juniper vs Arista command generation
8. Structured JSON responses
9. Streaming
10. Python + ping/DNS/HTTP tests + AI analysis
11. Eventually Netmiko/PyATS + AI

Keep your existing setup cell and reuse `client`:

```python
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

MODEL = "openrouter/free"
```

## Example 1 — Analyze `show interfaces`

This is one of the easiest practical examples.

```python
interface_output = """
Ethernet1/1 is up, line protocol is up
  MTU 9216 bytes
  5 minute input rate 850000000 bits/sec
  5 minute output rate 920000000 bits/sec
  125 input errors
  87 CRC errors
  0 frame errors
  15432 output drops
"""

prompt = f"""
You are a senior network engineer.

Analyze this interface output:

{interface_output}

Explain:
1. What looks normal
2. What looks abnormal
3. Possible causes
4. What I should check next
5. Give troubleshooting commands
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)
```

The AI should recognize things such as CRC errors, drops and high utilization.

---

# Example 2 — BGP troubleshooting assistant

This is much closer to real network troubleshooting.

```python
bgp_output = """
RP/0/RP0/CPU0:R1# show bgp summary

Neighbor        AS      MsgRcvd    MsgSent    State
10.1.1.2        65002   123455     125432     Established
10.1.1.6        65003   0          0          Active
10.1.1.10       65004   0          0          Idle
"""

prompt = f"""
Act as a senior Cisco IOS XR network engineer.

Analyze this BGP output:

{bgp_output}

For every neighbor explain:

- Current state
- Whether it is healthy
- Possible problem
- Commands to run next
- Most likely root causes

Give me a troubleshooting sequence.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)
```

You can make the data more complicated later:

```text
Idle
Connect
Active
OpenSent
OpenConfirm
Established
```

and see whether the model correctly explains the BGP FSM.

---

# Example 3 — Missing SYN-ACK troubleshooting

This relates directly to packet-loss troubleshooting.

```python
problem = """
Client: 10.1.1.10
Server: 10.20.20.50
Server Port: TCP 443

Packet capture on client:

10.1.1.10 -> 10.20.20.50 SYN
10.1.1.10 -> 10.20.20.50 SYN retransmission
10.1.1.10 -> 10.20.20.50 SYN retransmission

No SYN-ACK is visible in the client capture.
"""

prompt = f"""
You are a senior network troubleshooting engineer.

Analyze this problem:

{problem}

Explain possible causes including:

- Client
- Network
- Firewall
- NAT
- Load balancer
- Server
- Asymmetric routing
- Packet loss

Then provide a troubleshooting workflow from client to server.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)
```

This is a very good way to practice **layer-by-layer troubleshooting**.

---

# Example 4 — AI configuration reviewer

Give it a configuration and ask it to identify potential issues.

```python
config = """
router bgp 65001
 neighbor 10.1.1.2
  remote-as 65002
  update-source Loopback0

interface Loopback0
 ipv4 address 192.168.1.1 255.255.255.255
"""

prompt = f"""
Review this Cisco IOS XR-style configuration:

{config}

Look for:

1. BGP problems
2. Missing configuration
3. Reachability requirements
4. Update-source issues
5. Security concerns
6. Recommended verification commands

Do not assume the configuration is correct.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)
```

That last instruction is useful:

```text
Do not assume the configuration is correct.
```

Otherwise models sometimes become too agreeable with the supplied configuration.

---

# Example 5 — Cisco → Juniper → Arista command translator

This can be extremely useful while learning multiple vendors.

```python
question = """
I want to check:

- BGP summary
- BGP neighbor
- interface counters
- interface errors
- LACP status
- LLDP neighbors
- routing table
- ARP table
"""

prompt = f"""
You are a multi-vendor network engineer.

For the following requirements:

{question}

Give equivalent commands for:

Cisco IOS XR
Juniper Junos
Arista EOS

Create a comparison table.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)
```

This helps build your Cisco/Juniper/Arista knowledge simultaneously.

---

# Example 6 — Ask AI to create a troubleshooting decision tree

```python
prompt = """
Create a troubleshooting decision tree for:

BGP neighbor stuck in Active state.

Environment:
Cisco IOS XR

Start with Layer 1 and work upward.

Include:

Interface
ARP/ND
Routing
Ping
TCP 179
ACL/firewall
BGP configuration
ASN
Update source
Authentication
Timers

Show the troubleshooting sequence using ASCII arrows.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)
```

You may get something like:

```text
BGP Active
    |
    v
Interface UP?
    |
   YES
    |
    v
IP reachable?
    |
   YES
    |
    v
TCP/179 reachable?
    |
   YES
    |
    v
Check BGP configuration
    |
    +--> ASN correct?
    |
    +--> Update-source correct?
    |
    +--> Authentication correct?
```

---

# Example 7 — Analyze logs from a `.txt` file

Create:

```text
router_logs.txt
```

Paste some lab logs into it.

Then:

```python
with open("router_logs.txt", "r", encoding="utf-8") as file:
    logs = file.read()

prompt = f"""
Analyze these router logs:

{logs}

Identify:

- Interface flaps
- BGP changes
- ISIS/OSPF problems
- LACP problems
- BFD failures
- Packet loss indicators
- Important timestamps
- Probable root cause

Finally provide recommended troubleshooting commands.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a senior network reliability engineer."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)
```

For real production logs, remove confidential information before sending it to an external API.

---

# Example 8 — Network incident RCA generator

You can give the AI an incident timeline.

```python
incident = """
10:01:02 Interface Ethernet1/1 down
10:01:03 BFD neighbor down
10:01:04 ISIS adjacency down
10:01:05 BGP routes withdrawn
10:01:07 Traffic loss detected
10:01:12 Interface Ethernet1/1 up
10:01:13 BFD neighbor up
10:01:15 ISIS adjacency restored
10:01:20 Traffic recovered
"""

prompt = f"""
Analyze this network incident:

{incident}

Create:

1. Incident timeline
2. First failure
3. Cascading failures
4. Likely root cause
5. Customer impact
6. Recovery sequence
7. What monitoring should be added
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)
```

This teaches you an important distinction:

```text
Ethernet failure
      ↓
BFD failure
      ↓
ISIS failure
      ↓
Route withdrawal
      ↓
Traffic loss
```

The BGP/ISIS events might be **symptoms**, while the interface failure is closer to the root cause.

---

# Example 9 — Make the AI return JSON

This becomes important when you start building automation.

```python
import json

output = """
Ethernet1/1 is up
Input errors: 150
CRC errors: 120
Output drops: 25000
"""

prompt = f"""
Analyze:

{output}

Return ONLY valid JSON using this structure:

{{
    "interface_status": "",
    "severity": "",
    "problems": [],
    "possible_causes": [],
    "recommended_commands": []
}}
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

result = response.choices[0].message.content

print(result)
```

Instead of a paragraph, you're trying to get data such as:

```json
{
    "interface_status": "up",
    "severity": "high",
    "problems": [
        "CRC errors",
        "output drops"
    ],
    "possible_causes": [
        "physical layer issue",
        "congestion"
    ],
    "recommended_commands": [
        "show interfaces Ethernet1/1",
        "show interfaces counters errors"
    ]
}
```

This becomes powerful because Python can then act on individual fields.

---

# Example 10 — Python performs ping → AI analyzes result

Now we start combining **normal Python network testing + AI**.

```python
import subprocess

target = "8.8.8.8"

result = subprocess.run(
    ["ping", "-n", "4", target],
    capture_output=True,
    text=True
)

ping_output = result.stdout

print("PING OUTPUT")
print(ping_output)

prompt = f"""
You are a network engineer.

Analyze this Windows ping result:

{ping_output}

Determine:

- Reachability
- Packet loss
- Latency
- Any abnormal behavior
- Recommended next tests
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print("\nAI ANALYSIS")
print(response.choices[0].message.content)
```

Notice what's happening now:

```text
Python
   ↓
Ping 8.8.8.8
   ↓
Collect real result
   ↓
Send result to OpenRouter
   ↓
AI analysis
```

That's much more useful than simply asking the AI:

> Is 8.8.8.8 reachable?

The AI isn't guessing. **Python performs the test and the model interprets the evidence.**

---

# Example 11 — DNS test + AI

```python
import socket

website = "www.microsoft.com"

try:
    ip = socket.gethostbyname(website)
    result = f"DNS resolution successful. {website} resolved to {ip}"
except Exception as e:
    result = f"DNS resolution failed: {e}"

print(result)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": f"""
Analyze this DNS test:

{result}

Explain what it tells us and what network tests should be performed next.
"""
        }
    ]
)

print(response.choices[0].message.content)
```

---

# Example 12 — TCP port test + AI

Very useful for network engineers.

```python
import socket

host = "www.google.com"
port = 443

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.settimeout(5)

try:
    sock.connect((host, port))

    result = f"""
TCP connection successful.

Destination: {host}
Port: {port}
"""

except Exception as e:

    result = f"""
TCP connection failed.

Destination: {host}
Port: {port}

Error:
{e}
"""

finally:
    sock.close()

print(result)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": f"""
You are a network engineer.

Analyze this TCP connectivity test:

{result}

Explain what this proves about:

DNS
IP connectivity
TCP
Firewall
Destination service

Also explain what this test does NOT prove.
"""
        }
    ]
)

print(response.choices[0].message.content)
```

This last part is important:

> Also explain what this test does NOT prove.

It helps prevent the model from overclaiming.

---

# Example 13 — Build your own Network AI Assistant

Now combine everything into an interactive assistant:

```python
messages = [
    {
        "role": "system",
        "content": """
You are my senior network engineering assistant.

Specialties:

- Cisco IOS XR
- Cisco IOS XE
- Juniper Junos
- Arista EOS
- Azure Networking
- BGP
- OSPF
- ISIS
- BFD
- LACP
- MPLS
- EVPN/VXLAN
- TCP/IP
- Packet captures
- Network automation

When troubleshooting:

1. Identify symptoms.
2. Separate facts from assumptions.
3. Identify likely failure domain.
4. Give possible root causes.
5. Give verification commands.
6. Recommend troubleshooting order.
7. Do not claim a root cause without sufficient evidence.
"""
    }
]

while True:

    question = input("\nNetwork Engineer > ")

    if question.lower() in ["exit", "quit"]:
        break

    messages.append(
        {
            "role": "user",
            "content": question
        }
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    answer = response.choices[0].message.content

    print("\nAI Network Assistant:\n")
    print(answer)

    messages.append(
        {
            "role": "assistant",
            "content": answer
        }
    )
```

What's different here is that we keep:

```python
messages
```

instead of creating a brand-new conversation every time.

Therefore you can have a conversation like:

```text
Network Engineer >
My BGP neighbor is Active.

AI >
Check reachability and TCP/179...

Network Engineer >
Ping works.

AI >
Good. Next check TCP/179...

Network Engineer >
TCP/179 fails.

AI >
Now focus on ACL/firewall/BGP listener...
```

The model retains the previous messages.

## Where I would go next

The next major jump is to stop manually giving the AI information.

You can eventually build:

```text
             Network AI Agent
                    |
       +------------+------------+
       |            |            |
       v            v            v
     PING          DNS          TCP
       |            |            |
       +------------+------------+
                    |
                    v
                 Netmiko
                    |
          +---------+---------+
          |         |         |
          v         v         v
        Cisco    Juniper    Arista
          |         |         |
          +---------+---------+
                    |
                    v
             Collect outputs
                    |
                    v
               OpenRouter
                    |
                    v
             Analyze evidence
                    |
                    v
            Possible RCA +
       troubleshooting steps
```

For example, you could eventually type:

```text
Troubleshoot 10.1.1.20
```

and your Python program could automatically run:

```text
ping
DNS lookup
TCP test
traceroute
```

then send the collected evidence to the model for analysis.

After that, we can move into **Netmiko**, where your Python program connects to your lab Cisco/Juniper/Arista router, automatically runs `show interfaces`, `show bgp summary`, `show lacp`, etc., and sends the output through OpenRouter for troubleshooting. That would be a very useful next project once you're comfortable with the examples above.

In [7]:
from openrouter import OpenRouter
import os

with OpenRouter(api_key=os.getenv("OPENROUTER_API_KEY")) as client:
    response = client.chat.send(
        model="~openai/gpt-latest",
        messages=[
            {"role": "user", "content": "What is the meaning of life?"}
        ],
    )

    print(response.choices[0].message.content)

UnauthorizedResponseError: No cookie auth credentials found

In [1]:
import requests
import json

response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": "Bearer <OPENROUTER_API_KEY>",
    "HTTP-Referer": "<YOUR_SITE_URL>", # Optional. Site URL for rankings on openrouter.ai.
    "X-OpenRouter-Title": "<YOUR_SITE_NAME>", # Optional. Site title for rankings on openrouter.ai.
  },
  data=json.dumps({
    "model": "~openai/gpt-latest",
    "messages": [
      {
        "role": "user",
        "content": "What is the meaning of life?"
      }
    ]
  })
)

In [5]:
!pip install openrouter

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://packagefeedproxy.microsoft.io/pypi/simple/
     ---------------------------------------- 0.0/951.9 kB ? eta -:--:--
     ---------------------------------------- 951.9/951.9 kB 10.3 MB/s  0:00:00
     ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
     ---------------------------------------- 2.0/2.0 MB 39.1 MB/s  0:00:00

  Attempting uninstall: pydantic-core

    Found existing installation: pydantic_core 2.46.4

    Uninstalling pydantic_core-2.46.4:

      Successfully uninstalled pydantic_core-2.46.4

   ---------- ----------------------------- 1/4 [jsonpath-python]
  Attempting uninstall: pydantic
   ---------- ----------------------------- 1/4 [jsonpath-python]
    Found existing installation: pydantic 2.13.4
   ---------- ----------------------------- 1/4 [jsonpath-python]
    Uninstalling pydantic-2.13.4:
   ---------- ----------------------------- 1/4 [

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
paddlex 3.7.2 requires PyYAML==6.0.2, but you have pyyaml 6.0.3 which is incompatible.


In [6]:
from openrouter import OpenRouter
import os

with OpenRouter(api_key=os.getenv("OPENROUTER_API_KEY")) as client:
    response = client.chat.send(
        model="~openai/gpt-latest",
        messages=[
            {"role": "user", "content": "What is the meaning of life?"}
        ],
    )

    print(response.choices[0].message.content)

UnauthorizedResponseError: No cookie auth credentials found

In [8]:
from openai import OpenAI

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key="<OPENROUTER_API_KEY>",
)

completion = client.chat.completions.create(
  extra_headers={
    "HTTP-Referer": "<YOUR_SITE_URL>", # Optional. Site URL for rankings on openrouter.ai.
    "X-OpenRouter-Title": "<YOUR_SITE_NAME>", # Optional. Site title for rankings on openrouter.ai.
  },
  model="~openai/gpt-latest",
  messages=[
    {
      "role": "user",
      "content": "What is the meaning of life?"
    }
  ]
)

print(completion.choices[0].message.content)

AuthenticationError: Error code: 401 - {'error': {'message': 'Missing Authentication header', 'code': 401}}